# LangChain Memory 시리즈 1/7: ConversationBufferMemory

## 학습 목표

이 노트북을 마치면 다음을 할 수 있어요.

- LLM이 왜 태생적으로 대화를 기억하지 못하는지 설명할 수 있어요
- `ConversationBufferMemory`로 전체 대화 이력을 저장하고 조회할 수 있어요
- `save_context()` / `load_memory_variables()` / `clear()` API를 자유롭게 사용할 수 있어요
- `return_messages` 파라미터로 반환 형식을 선택할 수 있어요

## 사전 지식

| 개념 | 설명 |
|------|------|
| LLM (Large Language Model, 대규모 언어 모델) | 텍스트를 입력받아 텍스트를 생성하는 모델 |
| Stateless (무상태) | 요청이 끝나면 서버가 상태를 보존하지 않는 특성 |
| 프롬프트 (Prompt) | LLM에 전달하는 입력 텍스트 전체 |
| 토큰 (Token) | LLM이 텍스트를 처리하는 최소 단위 (대략 0.75 영단어) |

## LLM은 왜 기억을 못할까요?

LLM은 **Stateless(무상태)** 로 동작해요. 매 API 호출은 완전히 독립적이라서, 어제 나눈 대화를 오늘 다시 물어봐도 기억하지 못해요.

해결책은 간단해요. 대화 이력을 **프롬프트에 직접 포함**해서 LLM에게 전달하면 돼요. LangChain의 Memory 모듈이 바로 이 역할을 담당해요.

```mermaid
sequenceDiagram
    participant U as 사용자
    participant M as Memory
    participant LLM as LLM

    U->>M: "제주도 여행 알려줘"
    M->>LLM: [프롬프트] + [빈 이력]
    LLM-->>M: "3박 4일 추천드려요"
    M->>M: save_context() 저장

    U->>M: "렌터카 필요할까?"
    M->>LLM: [프롬프트] + [이전 대화 이력]
    LLM-->>M: "제주도라면 렌터카 추천해요"
    M->>M: save_context() 저장
```

> 🎯 **강의 포인트**: LLM의 Stateless 특성이 Memory 모듈의 존재 이유입니다. "LLM이 기억하는 것처럼 보이는 이유는 대화 이력을 프롬프트에 붙여서 보내기 때문"이라는 핵심 개념을 반드시 짚어주세요.

> **참고**: 이 노트북은 LangChain의 레거시(legacy) 메모리 클래스(`langchain.memory`)를 사용해요. LangChain 1.0.x에서는 `RunnableWithMessageHistory` + `ChatMessageHistory` 패턴이 권장되지만, 개념 학습에는 이 방식이 더 직관적이에요. 최신 방식은 10~11번 노트북을 참고하세요.

In [1]:
# ---------------------------------------------------
# 레거시 메모리 모듈 임포트
# ---------------------------------------------------
# ⚠️ LangChain 1.0.x에서는 더 이상 `langchain.memory` 패키지를 사용하지 않습니다.
#    구식 메모리 클래스를 시연하기 위해 `langchain.memory`에서 불러옵니다.
# 🎯 강의 포인트: 01~07번은 Legacy 패턴, 08~14번은 Modern 패턴입니다
from langchain.memory import ConversationBufferMemory

## 1. 기본 사용법 - 문자열로 저장 및 로드

가장 기본적인 사용법으로, 대화를 문자열 형태로 저장하고 로드합니다.


In [2]:
# ============================================================
# TODO: ConversationBufferMemory 인스턴스를 생성하고 첫 번째 대화를 저장하세요
# 힌트: ConversationBufferMemory() → save_context(inputs={"human": ...}, outputs={"ai": ...})
# 예상 결과: memory.load_memory_variables({})["history"]에 대화가 저장됨
# ============================================================

# 1단계: ConversationBufferMemory 인스턴스 생성
# - 기본값으로 생성 시 history 키에 문자열 형태로 대화 저장
memory = ConversationBufferMemory()

# 2단계: 첫 번째 대화 저장
# - inputs: 사용자 발화, outputs: AI 응답을 딕셔너리로 전달
memory.save_context(
    inputs={"human": "안녕하세요! 여행 계획을 세우고 싶어요. 제주도 여행을 추천해주세요."},
    outputs={"ai": "안녕하세요! 제주도 여행 계획을 도와드리겠습니다. 몇 박 며칠 일정을 생각하고 계신가요?"}
)

/var/folders/h6/vm46m4nx1xz4c_2phk_wz39h0000gn/T/ipykernel_87814/545391893.py:9: LangChainDeprecationWarning: Please see the migration guide at: https://python.langchain.com/docs/versions/migrating_memory/
  memory = ConversationBufferMemory()


### 저장된 대화 확인

`load_memory_variables()` 메서드를 사용하여 저장된 대화 기록을 확인할 수 있습니다.


In [3]:
# ---------------------------------------------------
# 저장된 대화 기록 조회
# ---------------------------------------------------
# load_memory_variables({}): 빈 딕셔너리를 전달하면 전체 이력 반환
# 반환값의 "history" 키에 문자열로 Human/AI 교대 발화가 담김
memory_variables = memory.load_memory_variables({})
print("=" * 60)
print("📝 저장된 대화 기록")
print("=" * 60)
print(memory_variables["history"])

📝 저장된 대화 기록
Human: 안녕하세요! 여행 계획을 세우고 싶어요. 제주도 여행을 추천해주세요.
AI: 안녕하세요! 제주도 여행 계획을 도와드리겠습니다. 몇 박 며칠 일정을 생각하고 계신가요?


### 여러 대화 추가하기

`save_context()` 메서드를 여러 번 호출하여 대화를 계속 추가할 수 있습니다.


In [4]:
# ---------------------------------------------------
# 여러 대화를 순차적으로 추가하여 이력 축적
# ---------------------------------------------------
# save_context()를 반복 호출하면 대화가 누적됨
# ⚠️ 주의: 버퍼 메모리는 대화 수에 제한이 없으므로 토큰 사용량이 계속 증가함
# 💡 팁: 대화가 10턴 이상 쌓이면 Buffer → Window → Token → Summary 순으로 교체를 고려하세요

# 두 번째 대화 추가
memory.save_context(
    inputs={"human": "3박 4일 일정으로 계획하고 있어요."},
    outputs={"ai": "좋은 선택이에요! 3박 4일이면 제주도의 주요 명소들을 여유롭게 둘러볼 수 있습니다. 첫날은 제주시 근처, 둘째 날은 중문관광단지, 셋째 날은 성산일출봉과 섭지코지, 마지막 날은 공항 근처를 추천드려요."}
)

# 세 번째 대화 추가
memory.save_context(
    inputs={"human": "렌터카를 빌릴까요, 아니면 대중교통을 이용할까요?"},
    outputs={"ai": "제주도는 렌터카가 가장 편리합니다. 특히 3박 4일 일정이라면 렌터카로 이동하시면 시간을 효율적으로 사용하실 수 있어요. 다만, 성수기에는 렌터카 예약이 빠르게 마감되니 미리 예약하시는 것을 추천드립니다."}
)

In [5]:
# ---------------------------------------------------
# 전체 대화 기록 확인
# ---------------------------------------------------
# 누적된 3개 대화 턴이 모두 history에 포함됨
print("=" * 60)
print("📚 전체 대화 기록")
print("=" * 60)
print(memory.load_memory_variables({})["history"])

📚 전체 대화 기록
Human: 안녕하세요! 여행 계획을 세우고 싶어요. 제주도 여행을 추천해주세요.
AI: 안녕하세요! 제주도 여행 계획을 도와드리겠습니다. 몇 박 며칠 일정을 생각하고 계신가요?
Human: 3박 4일 일정으로 계획하고 있어요.
AI: 좋은 선택이에요! 3박 4일이면 제주도의 주요 명소들을 여유롭게 둘러볼 수 있습니다. 첫날은 제주시 근처, 둘째 날은 중문관광단지, 셋째 날은 성산일출봉과 섭지코지, 마지막 날은 공항 근처를 추천드려요.
Human: 렌터카를 빌릴까요, 아니면 대중교통을 이용할까요?
AI: 제주도는 렌터카가 가장 편리합니다. 특히 3박 4일 일정이라면 렌터카로 이동하시면 시간을 효율적으로 사용하실 수 있어요. 다만, 성수기에는 렌터카 예약이 빠르게 마감되니 미리 예약하시는 것을 추천드립니다.


## 2. 메시지 객체로 저장 및 로드

`return_messages=True`로 설정하면 `HumanMessage`와 `AIMessage` 객체를 반환합니다.  
이 방식은 `ChatPromptTemplate`과 함께 사용할 때 유용합니다.


In [6]:
# ============================================================
# TODO: return_messages=True 옵션으로 메모리를 생성하고 대화를 저장하세요
# 힌트: ConversationBufferMemory(return_messages=True)
# 예상 결과: load_memory_variables({})["history"]가 메시지 객체 리스트로 반환됨
# ============================================================

# 1단계: return_messages=True로 메모리 생성
# - return_messages=True: 문자열 대신 HumanMessage/AIMessage 객체 리스트 반환
# - ChatPromptTemplate과 직접 연결할 때 이 형식이 필요
memory_with_messages = ConversationBufferMemory(return_messages=True)

# 2단계: 대화 저장
memory_with_messages.save_context(
    inputs={"human": "안녕하세요! 여행 계획을 세우고 싶어요. 제주도 여행을 추천해주세요."},
    outputs={"ai": "안녕하세요! 제주도 여행 계획을 도와드리겠습니다. 몇 박 며칠 일정을 생각하고 계신가요?"}
)

memory_with_messages.save_context(
    inputs={"human": "3박 4일 일정으로 계획하고 있어요."},
    outputs={"ai": "좋은 선택이에요! 3박 4일이면 제주도의 주요 명소들을 여유롭게 둘러볼 수 있습니다."}
)

In [7]:
# ---------------------------------------------------
# 메시지 객체 형태로 반환된 대화 기록 확인
# ---------------------------------------------------
# type(msg).__name__: HumanMessage 또는 AIMessage 클래스명 출력
# msg.content: 실제 발화 텍스트
messages = memory_with_messages.load_memory_variables({})["history"]
print("=" * 60)
print("💬 메시지 객체 형태의 대화 기록")
print("=" * 60)
for i, msg in enumerate(messages, 1):
    print(f"{i}. {type(msg).__name__}: {msg.content}")

💬 메시지 객체 형태의 대화 기록
1. HumanMessage: 안녕하세요! 여행 계획을 세우고 싶어요. 제주도 여행을 추천해주세요.
2. AIMessage: 안녕하세요! 제주도 여행 계획을 도와드리겠습니다. 몇 박 며칠 일정을 생각하고 계신가요?
3. HumanMessage: 3박 4일 일정으로 계획하고 있어요.
4. AIMessage: 좋은 선택이에요! 3박 4일이면 제주도의 주요 명소들을 여유롭게 둘러볼 수 있습니다.


## 3. 메모리 키 커스터마이징

`memory_key` 파라미터를 사용하여 메모리 변수의 키 이름을 변경할 수 있습니다.


In [8]:
# ---------------------------------------------------
# memory_key 파라미터로 반환 키 이름 커스터마이징
# ---------------------------------------------------
# memory_key="chat_history": load_memory_variables()가 "history" 대신 "chat_history" 키를 반환
# 체인 프롬프트 변수명이 "history"가 아닐 때 이 파라미터로 맞춰줘야 함
memory_custom = ConversationBufferMemory(memory_key="chat_history")

memory_custom.save_context(
    inputs={"human": "안녕하세요!"},
    outputs={"ai": "안녕하세요! 무엇을 도와드릴까요?"}
)

# 'history' 대신 'chat_history' 키로 접근
memory_vars = memory_custom.load_memory_variables({})
print("=" * 60)
print("🔑 커스텀 메모리 키 사용")
print("=" * 60)
print(f"사용 가능한 키: {list(memory_vars.keys())}")
print(f"\n대화 기록:\n{memory_vars['chat_history']}")

🔑 커스텀 메모리 키 사용
사용 가능한 키: ['chat_history']

대화 기록:
Human: 안녕하세요!
AI: 안녕하세요! 무엇을 도와드릴까요?


## 4. 실전 예제: 여행 계획 챗봇

여러 대화를 저장하고, 이전 대화 내용을 기억하는 간단한 챗봇 예제입니다.


In [9]:
# ============================================================
# TODO: 4개의 대화를 ConversationBufferMemory에 순서대로 저장하세요
# 힌트: for 루프로 conversations 리스트를 순회하며 save_context() 호출
# 예상 결과: travel_memory.load_memory_variables({})["history"]에 4턴 모두 포함
# ============================================================

# 1단계: 새로운 메모리 인스턴스 생성
travel_memory = ConversationBufferMemory()

# 2단계: 여행 계획 대화 데이터 정의
conversations = [
    {
        "human": "안녕하세요! 곧 휴가를 가는데 여행지 추천을 받고 싶어요.",
        "ai": "안녕하세요! 휴가 계획을 도와드리겠습니다. 어떤 스타일의 여행을 선호하시나요? 해변 휴양, 도시 탐방, 자연 속 힐링 중에서 선택해주세요."
    },
    {
        "human": "해변 휴양을 좋아해요. 바다를 보면서 쉬고 싶어요.",
        "ai": "해변 휴양이라면 제주도나 부산이 좋을 것 같아요. 제주도는 한적한 해변과 카페가 많고, 부산은 해운대와 광안리 해수욕장이 유명합니다. 어느 쪽을 선호하시나요?"
    },
    {
        "human": "제주도로 가고 싶어요. 3박 4일 일정으로 계획하고 있어요.",
        "ai": "제주도 3박 4일 일정이시군요! 첫날은 제주시 근처에서 휴식을 취하고, 둘째 날은 중문관광단지와 천지연폭포를 방문하시는 것을 추천드려요. 셋째 날은 성산일출봉과 섭지코지, 마지막 날은 공항 근처에서 쇼핑을 즐기시면 좋을 것 같습니다."
    },
    {
        "human": "렌터카를 빌려야 할까요?",
        "ai": "네, 제주도는 렌터카가 가장 편리합니다. 특히 여러 명소를 둘러보실 계획이시라면 렌터카를 추천드려요. 다만 성수기에는 미리 예약하시는 것이 좋습니다."
    }
]

# 3단계: 모든 대화를 순서대로 저장
for conv in conversations:
    travel_memory.save_context(
        inputs={"human": conv["human"]},
        outputs={"ai": conv["ai"]}
    )

In [10]:
# ---------------------------------------------------
# 저장된 전체 여행 계획 대화 기록 출력
# ---------------------------------------------------
print("=" * 60)
print("🗺️  여행 계획 대화 기록")
print("=" * 60)
history = travel_memory.load_memory_variables({})["history"]
print(history)

🗺️  여행 계획 대화 기록
Human: 안녕하세요! 곧 휴가를 가는데 여행지 추천을 받고 싶어요.
AI: 안녕하세요! 휴가 계획을 도와드리겠습니다. 어떤 스타일의 여행을 선호하시나요? 해변 휴양, 도시 탐방, 자연 속 힐링 중에서 선택해주세요.
Human: 해변 휴양을 좋아해요. 바다를 보면서 쉬고 싶어요.
AI: 해변 휴양이라면 제주도나 부산이 좋을 것 같아요. 제주도는 한적한 해변과 카페가 많고, 부산은 해운대와 광안리 해수욕장이 유명합니다. 어느 쪽을 선호하시나요?
Human: 제주도로 가고 싶어요. 3박 4일 일정으로 계획하고 있어요.
AI: 제주도 3박 4일 일정이시군요! 첫날은 제주시 근처에서 휴식을 취하고, 둘째 날은 중문관광단지와 천지연폭포를 방문하시는 것을 추천드려요. 셋째 날은 성산일출봉과 섭지코지, 마지막 날은 공항 근처에서 쇼핑을 즐기시면 좋을 것 같습니다.
Human: 렌터카를 빌려야 할까요?
AI: 네, 제주도는 렌터카가 가장 편리합니다. 특히 여러 명소를 둘러보실 계획이시라면 렌터카를 추천드려요. 다만 성수기에는 미리 예약하시는 것이 좋습니다.


## 5. 메모리 초기화

`clear()` 메서드를 사용하여 저장된 모든 대화 기록을 삭제할 수 있습니다.


In [11]:
# ---------------------------------------------------
# clear()로 메모리 초기화
# ---------------------------------------------------
# clear(): 저장된 모든 대화를 삭제하고 history를 빈 문자열로 초기화
# 새 대화 세션을 시작하거나 테스트를 리셋할 때 사용

# 초기화 전 크기 확인
print("=" * 60)
print("🗑️  메모리 초기화 전")
print("=" * 60)
print(f"대화 기록 길이: {len(travel_memory.load_memory_variables({})['history'])} 문자")

# 메모리 초기화
travel_memory.clear()

# 초기화 후 확인
print("\n" + "=" * 60)
print("✅ 메모리 초기화 후")
print("=" * 60)
memory_after_clear = travel_memory.load_memory_variables({})
print(f"대화 기록: {memory_after_clear['history']}")
print(f"대화 기록 길이: {len(memory_after_clear['history'])} 문자")

🗑️  메모리 초기화 전
대화 기록 길이: 535 문자

✅ 메모리 초기화 후
대화 기록: 
대화 기록 길이: 0 문자


## 💡 핵심 정리

```python
# ConversationBufferMemory 기본 사용법
memory = ConversationBufferMemory()

# 대화 저장
memory.save_context(
    inputs={"human": "사용자 메시지"},
    outputs={"ai": "AI 응답"}
)

# 대화 로드
history = memory.load_memory_variables({})["history"]

# 메모리 초기화
memory.clear()
```

### 주요 특징
- ✅ **전체 대화 저장**: 모든 대화 턴을 순서대로 저장
- ✅ **간단한 API**: `save_context()`, `load_memory_variables()`, `clear()`
- ✅ **유연한 형식**: 문자열 또는 메시지 객체로 반환 가능
- ✅ **짧은 대화에 적합**: 토큰 제한이 크지 않은 경우 사용

### 주의사항
- ⚠️ **토큰 제한**: 대화가 길어질수록 토큰 사용량이 증가
- ⚠️ **메모리 사용**: 모든 대화를 저장하므로 메모리 사용량 증가
- ⚠️ **긴 대화 부적합**: 매우 긴 대화에서는 다른 메모리 타입 고려 필요

> 🔑 **핵심 개념**: Legacy(01~07) 메모리는 학습/프로토타입용이고, Modern(08~14) 패턴이 실제 서비스에서 권장됩니다. 이 두 세계를 연결하는 스토리라인을 기억하세요!